# sim_dual_arm — Google Colab Pro ACT 训练流程

本 Notebook 通过 **Google Colab VS Code 插件** 连接云端 GPU 执行 **ACT 模型训练**，使用 **Google Drive 作为中转存储**，采用先在本机完成紧凑视频转换、再上传到云端训练的低流量路线。

## 前置步骤（在本机执行）

1. 在本机完成正式数据采集，并在本机 `learning` 容器内先将原始帧转为紧凑的 H.264 + Parquet 格式（大幅减小上传体积）：
   ```bash
   ./env.sh exec learning scripts/convert.sh --robot <robot> --task <task>
   ```
2. 在本机仓库根目录打包转换后的 `lerobot/` 目录：
   ```bash
   tar -C var/<robot>/<task>/datasets -cf lerobot.tar lerobot
   ```
3. 将 `lerobot.tar` 上传到 Google Drive 的对应目录（默认 `MyDrive/sim_dual_arm/<robot>/<task>/datasets/lerobot.tar`）。

## 使用步骤

1. 在 VS Code 右上角点击 **Select Kernel** → **Colab** → 连接带有 Pro GPU（如 **L4** / **A100**）的运行时。
2. 在下方 **第 1 节** 修改 `ROBOT`、`TASK` 与训练模式。
3. 按顺序运行全部 Cell；训练完成后模型与日志会自动同步回 Google Drive。


In [ ]:
# 1. 参数配置（按需修改后运行）
import os

REPO_URL = "https://github.com/ljk4/sim_dual_arm.git"
REPO_BRANCH = "main"
WORK_DIR = "/content/sim_dual_arm"
DRIVE_ROOT = "/content/drive/MyDrive/sim_dual_arm"

# 按需切换任务：如 marvin_sim / stack 或 marvin_sim / handover
ROBOT = "marvin_sim"
TASK = "stack"
TRAIN_MODE = "full"   # 可选: smoke / overfit / full
MAX_STEPS = None      # None 使用配方默认步数；可设整数如 5000
DATASET_ARCHIVE = "lerobot.tar"
USE_DRIVE_CACHE = True
TORCH_INDEX_URL = ""  # 留空自动按显卡驱动选择 cu128 / cu126

os.environ.update({
    "SDA_REPO_URL": REPO_URL,
    "SDA_REPO_BRANCH": REPO_BRANCH,
    "SDA_WORK_DIR": WORK_DIR,
    "SDA_DRIVE_ROOT": DRIVE_ROOT,
    "SDA_ROBOT": ROBOT,
    "SDA_TASK": TASK,
    "SDA_TRAIN_MODE": TRAIN_MODE,
    "SDA_MAX_STEPS": "" if MAX_STEPS is None else str(int(MAX_STEPS)),
    "SDA_DATASET_ARCHIVE": DATASET_ARCHIVE,
    "SDA_USE_DRIVE_CACHE": "1" if USE_DRIVE_CACHE else "0",
    "SDA_TORCH_INDEX_URL": TORCH_INDEX_URL,
})
print(f"配置已就绪: {ROBOT}/{TASK} | mode={TRAIN_MODE} | max_steps={MAX_STEPS}")

In [ ]:
# 2. 检查 GPU 并挂载 Google Drive
import subprocess
from pathlib import Path
from google.colab import drive

subprocess.run(["nvidia-smi"], check=False)
drive.mount("/content/drive", force_remount=False)

drive_base = Path(DRIVE_ROOT)
for directory in (
    drive_base / "_cache",
    drive_base / ROBOT / TASK / "datasets",
    drive_base / ROBOT / TASK / "models",
    drive_base / ROBOT / TASK / "logs" / "training",
):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Google Drive 中转目录已就绪: {drive_base / ROBOT / TASK}")

In [ ]:
%%bash
# 3. 克隆或更新公开仓库到 Colab 本地盘
set -euo pipefail

if [[ -d "$SDA_WORK_DIR/.git" ]]; then
  echo "仓库已存在，更新分支 $SDA_REPO_BRANCH..."
  git -C "$SDA_WORK_DIR" fetch origin "$SDA_REPO_BRANCH"
  git -C "$SDA_WORK_DIR" checkout "$SDA_REPO_BRANCH"
  git -C "$SDA_WORK_DIR" pull --ff-only origin "$SDA_REPO_BRANCH"
else
  echo "克隆公开仓库 $SDA_REPO_URL ($SDA_REPO_BRANCH) 到 $SDA_WORK_DIR..."
  git clone --branch "$SDA_REPO_BRANCH" --single-branch "$SDA_REPO_URL" "$SDA_WORK_DIR"
fi

echo -n "当前提交: "
git -C "$SDA_WORK_DIR" log -1 --oneline

In [ ]:
%%bash
# 4. 安装学习端环境与第三方源码/权重
set -euo pipefail
cd "$SDA_WORK_DIR"

args=("--repo-root" "$SDA_WORK_DIR")
if [[ "$SDA_USE_DRIVE_CACHE" == "1" ]]; then
  args+=("--cache-dir" "$SDA_DRIVE_ROOT/_cache")
fi
if [[ -n "$SDA_TORCH_INDEX_URL" ]]; then
  args+=("--torch-index-url" "$SDA_TORCH_INDEX_URL")
fi

bash docs/colab/setup_colab_learning.sh "${args[@]}"

In [ ]:
# 5. 从 Google Drive 拉取数据集到 Colab 本地盘
import json
import shutil
import subprocess
from pathlib import Path

drive_datasets = Path(DRIVE_ROOT) / ROBOT / TASK / "datasets"
local_datasets = Path(WORK_DIR) / "var" / ROBOT / TASK / "datasets"
local_datasets.mkdir(parents=True, exist_ok=True)

target_dir = local_datasets / "lerobot"
staging_dir = local_datasets / ".unpack_tmp"
if staging_dir.exists():
    shutil.rmtree(staging_dir)
staging_dir.mkdir(parents=True, exist_ok=True)

archive_src = drive_datasets / DATASET_ARCHIVE
unpacked_src = drive_datasets / "lerobot"
tmp_archive = Path("/tmp") / f"{ROBOT}_{TASK}_{DATASET_ARCHIVE}"

try:
    if archive_src.is_file():
        print(f"正在从 Drive 复制压缩包到 /tmp: {archive_src}")
        shutil.copy2(archive_src, tmp_archive)
        print(f"正在解压到本地盘: {staging_dir}")
        subprocess.run(["tar", "-xf", str(tmp_archive), "-C", str(staging_dir)], check=True)
        if (staging_dir / "lerobot" / "manifest.json").is_file():
            extracted_root = staging_dir / "lerobot"
        elif (staging_dir / "manifest.json").is_file():
            extracted_root = staging_dir
        else:
            raise FileNotFoundError(
                f"{archive_src} 解压后未找到 manifest.json（应在根级或 lerobot/ 下）"
            )
    elif (unpacked_src / "manifest.json").is_file():
        print(f"未找到 {archive_src.name}，改用 Drive 上的已解压目录: {unpacked_src}")
        extracted_root = staging_dir / "lerobot"
        shutil.copytree(unpacked_src, extracted_root)
    else:
        raise FileNotFoundError(
            f"在 Google Drive 未找到数据集，请先上传 {archive_src} 或目录 {unpacked_src}"
        )

    if target_dir.exists():
        shutil.rmtree(target_dir)
    shutil.move(str(extracted_root), str(target_dir))
finally:
    if tmp_archive.exists():
        tmp_archive.unlink()
    if staging_dir.exists():
        shutil.rmtree(staging_dir, ignore_errors=True)

required = ("manifest.json", "train", "validation", "processors")
missing = [name for name in required if not (target_dir / name).exists()]
if missing:
    raise RuntimeError(f"本地数据集目录缺少必要项: {missing} ({target_dir})")

manifest = json.loads((target_dir / "manifest.json").read_text(encoding="utf-8"))
splits = manifest.get("splits", {})
tr = splits.get("training", {})
va = splits.get("validation", {})
print(f"本地数据集就绪: {target_dir}")
print(
    f"status={manifest.get('status')} | synthetic={manifest.get('synthetic')} | "
    f"train={tr.get('episode_count')} 条/{tr.get('frame_count')} 帧 | "
    f"validation={va.get('episode_count')} 条/{va.get('frame_count')} 帧"
)

In [ ]:
# 6. 启动 ACT 训练（自动检测 BF16 支持）
import shlex
import subprocess
from pathlib import Path

work = Path(WORK_DIR)
venv_python = work / ".venv" / "bin" / "python"

bf16_probe = subprocess.run(
    [
        str(venv_python),
        "-c",
        "import torch; print('1' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else '0')",
    ],
    check=True,
    capture_output=True,
    text=True,
)
supports_bf16 = bf16_probe.stdout.strip() == "1"

train_args = ["bash", "learning/lerobot/scripts/train.sh"]
if supports_bf16:
    print("当前 GPU 支持硬件 BF16，直接使用方案默认训练配方。")
    train_args.extend(["--robot", ROBOT, "--task", TASK])
else:
    print("当前 GPU 不支持硬件 BF16，生成临时 FP16 配置 (amp_dtype=float16, num_workers=2)...")
    src_cfg = work / "shared" / "scenarios" / ROBOT / TASK / "configs" / "learning" / "act_training.yaml"
    debug_dir = work / "var" / ROBOT / TASK / "debug"
    debug_dir.mkdir(parents=True, exist_ok=True)
    fp16_cfg = debug_dir / "act_training_colab_fp16.yaml"
    subprocess.run(
        [
            str(venv_python),
            "-c",
            (
                "import sys, yaml; "
                "doc = yaml.safe_load(open(sys.argv[1], encoding='utf-8')); "
                "doc['training']['amp_dtype'] = 'float16'; "
                "doc['training']['num_workers'] = 2; "
                "yaml.safe_dump(doc, open(sys.argv[2], 'w', encoding='utf-8'), sort_keys=False, allow_unicode=True)"
            ),
            str(src_cfg),
            str(fp16_cfg),
        ],
        check=True,
    )
    train_args.extend(["--config", str(fp16_cfg)])

train_args.extend(["--mode", TRAIN_MODE])
if MAX_STEPS is not None:
    train_args.extend(["--max-steps", str(int(MAX_STEPS))])

cmd_str = "cd " + shlex.quote(str(work)) + " && source .env.learning && exec " + shlex.join(train_args)
print("执行命令:", shlex.join(train_args), flush=True)

proc = subprocess.Popen(
    ["bash", "-lc", cmd_str],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise subprocess.CalledProcessError(ret, train_args)

In [ ]:
# 7. 同步最新模型与训练日志回 Google Drive
import shutil
from pathlib import Path

local_var = Path(WORK_DIR) / "var" / ROBOT / TASK
models_dir = local_var / "models"
candidates = sorted(
    (p for p in models_dir.iterdir() if p.is_dir() and p.name.startswith("act_")),
    key=lambda p: p.name,
    reverse=True,
)
if not candidates:
    raise FileNotFoundError(f"未找到训练产出的模型目录: {models_dir}")

latest_model = candidates[0]
if not (latest_model / "model.safetensors").is_file():
    raise FileNotFoundError(f"模型目录缺少 model.safetensors: {latest_model}")

drive_task_root = Path(DRIVE_ROOT) / ROBOT / TASK
dst_model = drive_task_root / "models" / latest_model.name
shutil.copytree(latest_model, dst_model, dirs_exist_ok=True)

src_log = local_var / "logs" / "training" / latest_model.name
dst_log = drive_task_root / "logs" / "training" / latest_model.name
if src_log.is_dir():
    shutil.copytree(src_log, dst_log, dirs_exist_ok=True)

print(f"模型已同步至 Google Drive: {dst_model}")
for item in sorted(dst_model.iterdir()):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  - {item.name}: {size_mb:.2f} MiB")
if dst_log.is_dir():
    print(f"训练日志已同步至 Google Drive: {dst_log}")

print("\n下载回本机推理请放入:")
print(f"  var/{ROBOT}/{TASK}/models/{latest_model.name}/")